In [20]:
from dotenv import load_dotenv
import pandas as pd
from processing import (
    generate_comments,
    df_character_cleaning,
    create_entities_df,
    create_movie_info,
    get_comments_sentiment,
)
from app import (
    extract_youtube_id,
    extract_imdb_id,
#    process_comments_in_batches,
)
from topic_summarization import (
    comments_summarizer,
    Claude,
    ChatGPT,
    comment_averaging,
#    match_topics_comments,
)
import json
import concurrent.futures
from functools import partial
import numpy as np

def summarize_comments(df, movie_info_str):
    all_resp = comments_summarizer(df, movie_info_str, Claude)
    resp_list = [item for item in all_resp.splitlines() if item]
    return resp_list

load_dotenv()

True

In [2]:
## VENOM
# youtube_ref = "https://www.youtube.com/watch?v=HyIyd9joTTc"
# video_id = extract_youtube_id(youtube_ref)

tiktok = pd.read_csv('Venom_comments.csv')
tiktok = tiktok["Comment"].to_list()

imdb_ref = "https://www.imdb.com/title/tt16366836/"
movie_id = extract_imdb_id(imdb_ref)

In [3]:
#comments = generate_comments(video_id, os.getenv("YT_KEY"), max_comments=1000)
comments = tiktok
comments = df_character_cleaning(comments)

2024-09-19 14:22:23.784 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [4]:
_, entities_df = create_entities_df(movie_id)

In [5]:
movie_info = create_movie_info(movie_id, entities_df)

In [6]:
topics = summarize_comments(comments, movie_info)

1. Excitement and anticipation for seeing the powerful symbiote god Knull make his live-action debut as the main villain.
2. Appreciation for the humor and banter between Eddie Brock and Venom.
3. Interest in seeing how the symbiote storyline and Eddie/Venom's journey concludes in this final film of the trilogy.
4. Praise for Tom Hardy's performance and commitment to the Venom character. 
5. Enjoyment of the action, visuals, and over-the-top moments shown in the trailer.
6. Concerns that Knull will be nerfed or underutilized given his immense power level in the comics.
7. Complaints that Venom has been made too comedic and not portrayed as a serious, threatening character.
8. Disappointment over the lack of Spider-Man being involved in this Venom trilogy.
9. Skepticism over the quality of the story and screenplay based on the previous Venom films.
10. Criticism of the CGI and visual effects looking unpolished or unconvincing at times.


In [56]:
def process_comments_in_batches(comments, summary, _match_fctn, batch_size=50):
    batches = []
    for i in range(0, len(comments), batch_size):
        batches.append(comments[i : i + batch_size])
    with concurrent.futures.ThreadPoolExecutor() as executor:
        results = list(executor.map(partial(_match_fctn, all_resp=summary), batches))
    flattened_result = [
        item for sublist in results if sublist is not None for item in sublist
    ]
    df = pd.DataFrame(flattened_result)
    return df

def match_topics_comments(text, all_resp):
    print("matching in progress")
    " \n".join(t for t in text)
    topic_analysis_prompt = f"""The following statements represent general expressed themes associated with a set of movie trailer comments \n {all_resp} \n
  You will be given a set of comments concerning the same movie trailer. For each comment, I would like you
  to ouput the original, unedited comment, along with indicators for each comment topic. If the comment relates to the topic,
  you will assign it a 1, otherwise you will assign it a 0. You will output this as a list in JSON format. 

  For example, consider these 3 positive and 2 negative topics concerning the trailer for The Social Network:
  1. Overwhelming admiration for the quality of the film and trailer with regards to storytelling, presentation and overall execution. 
  2. Great appreciation for Director David Finchers directing prowess and his depiction of Facebooks rise, resonating with societal themes. 
  3. Highly appreciated performances from the cast, with special mentions of actors such as Andrew Garfield and Jesse Eisenberg.

  4. Viewer disapproval of Facebook as a platform and its societal impact, potentially skewing their perception of the film negatively. 
  5. Criticisms on historical inaccuracy in the portrayal of Facebooks inception and portrayal of Mark Zuckerberg. 
  
  And these comments:
  I hate Facebook and I love this movie 
  I always come back to this movie. Theres nothing like it. Every time I re watch it, theres always something I noice that I didnt the last time. Its art. And the way its created is perfect. 
  A special movie dedicated to founders of the Facebook and what did went inside their friendship through the process of creating the worlds dominant mass reaching communication forum. Acted perfectly by Andrew and Jesse its a definite watch for audiences across the world. 
  just rewatched the film last night - even if its not 100% accurate, its a masterpiece of filmmaking, sound design, cinematography. 
  Lex Luthor created Facebook. 
  A lot of people are talking about how great the acting is, but I do not buy it. This movie is carried by the filmmakers behind the camera, even though the story is made-up. 
  He's smart, but I don't trust him. 


  The output would be:
    {
    [
    {
  "0" : "I hate Facebook and I love this movie",
  "1" : 1,
  "2" : 0,
  "3" : 0,
  "4" : 1,
  "5" : 0
    },
    {
  "0" : "I always come back to this movie. Theres nothing like it. Every time I re watch it, theres always something I noice that I didnt the last time. Its art. And the way its created is perfect.",
  "1" : 1,
  "2" : 0,
  "3" : 0,
  "4" : 0,
  "5" : 0
    },
    {
  "0" : "A special movie dedicated to founders of the Facebook and what did went inside their friendship through the process of creating the worlds dominant mass reaching communication forum. Acted perfectly by Andrew and Jesse its a definite watch for audiences across the world.",
  "1" : 1,
  "2" : 0,
  "3" : 1,
  "4" : 0,
  "5" : 0
    },
    {
  "0" : "just rewatched the film last night - even if its not 100% accurate, its a masterpiece of filmmaking, sound design, cinematography.",
  "1" : 1,
  "2" : 0,
  "3" : 0,
  "4" : 0,
  "5" : 1
    },
    {
  "0" : "Lex Luthor created Facebook.",
  "1" : 0,
  "2" : 0,
  "3" : 0,
  "4" : 0,
  "5" : 0
    },
    {
  "0" : "A lot of people are talking about how great the acting is, but I do not buy it. This movie is carried by the filmmakers behind the camera, even though the story is made-up.",
  "1" : 1,
  "2" : 1,
  "3" : 0,
  "4" : 0,
  "5" : 1

    },
    {
  "0" : "He's smart, but I don't trust him.",
  "1" : 0,
  "2" : 0,
  "3" : 0,
  "4" : 0,
  "5" : 0
    }
    ]
    }, where 0 is attributed to the comment, 1 is attributed to the first theme, 2 to the second theme, 3 to the third, etc. 
    
    Note that the topics you will be given can be positive or negative, and will be 10 in total. This example is for illustrative purposes only. 

    Only classify the comment if it directly relates to the respective theme; some comments may not belong to any topic, in which case you will generate 0 for each value of the key-value pairs.

    For example, the comments 'Lex Luthor created Facebook' and 'He's smart, but I don't trust him' are both related to the trailer, but are not specific enough to fit in any category.

    Additionally, although the comment 'A lot of people are talking about how great the acting is, but I do not buy it. This movie is carried by the filmmakers behind the camera, even though the story is made-up.' mentions the praise for the acting, the comment itself is not praiseworth, so it is not attributed to the
    topic 'Highly appreciated performances from the cast, with special mentions of actors such as Andrew Garfield and Jesse Eisenberg'.
.
Please output in the same format for these comments {text} and the provided themes: {all_resp}. DO NOT output any other text other than the information specified and DO NOT use space brackets '()' or apostrophes like '. Please generate the full comment. It is extremely important that you fully follow these instructions:
"""
    try:
        chat = ChatGPT(
            system_message=f"""You are an expert comment analyzer who outputs in JSON format. You are not allowed to use any apostrophes (') in your generation. Simply use double quotes("") 
                                    instead; only use single-quotes ('') inside double-quotes if necessary, never use double-quotes within double-quotes. 
                                    You will be prompted with many comments; please perform the analsys for every single comment, do not skip any even though it may be computationally expensive. 
                                    Please generate the entire comment in your analysis, and only classify the comment if it directly relates to the respective theme, this is extremely important!"""
        )
        chat.add_user_message(topic_analysis_prompt)
        summarized_chunk = chat.get_response()
        try:
          summarized_chunk = json.loads(summarized_chunk)
          print("matching done")
          return summarized_chunk
        except Exception as e:
          print(f"Error during json: {e}")
          print(summarized_chunk)
          return
        
    except Exception as e:
        print(f"Error during matching: {e}")
        return None

In [97]:
topics

['1. Excitement and anticipation for seeing the powerful symbiote god Knull make his live-action debut as the main villain.',
 '2. Appreciation for the humor and banter between Eddie Brock and Venom.',
 "3. Interest in seeing how the symbiote storyline and Eddie/Venom's journey concludes in this final film of the trilogy.",
 "4. Praise for Tom Hardy's performance and commitment to the Venom character. ",
 '5. Enjoyment of the action, visuals, and over-the-top moments shown in the trailer.',
 '6. Concerns that Knull will be nerfed or underutilized given his immense power level in the comics.',
 '7. Complaints that Venom has been made too comedic and not portrayed as a serious, threatening character.',
 '8. Disappointment over the lack of Spider-Man being involved in this Venom trilogy.',
 '9. Skepticism over the quality of the story and screenplay based on the previous Venom films.',
 '10. Criticism of the CGI and visual effects looking unpolished or unconvincing at times.']

In [98]:
comments_topics_df = process_comments_in_batches(comments,topics, match_topics_comments, batch_size=50)

matching in progressmatching in progress
matching in progress

matching in progress
matching in progress
matching in progress
matching in progress
matching in progress
matching in progress
matching in progress
matching in progress
matching in progress
matching done
matching in progress
matching done
matching in progress
matching done
matching in progress
matching done
matching in progress
matching done
matching in progress
matching done
matching in progress
matching done
matching in progress
matching done
matching in progress
matching done
matching done
matching done
matching done
matching done
matching done
matching done
matching done
matching done
matching done
matching done
matching done


In [99]:
comments_topics_df

,0,1,2,3,4,5,6,7,8,9,10
0,"If you and Venom stay together, the world will endVenom: LETS GO GAMBLING!! AWW DANG IT!!!",0,1,0,0,1,0,0,0,0,0
1,Every time someone likes my comment Ill watch it again,0,0,0,0,0,0,0,0,0,0
2,HOLY SHIT! NEVER THOUGHT KNULL IN LIVE ACTION WAS POSSIBLE BUT HERE WE ARE,1,0,0,0,0,0,0,0,0,0
3,How does Tom Cruise do this :skull:Tom Cruise: I do my own stunts,0,0,0,0,0,0,0,0,0,0
4,Venom at the start of the trailer: Very seriousVenom at the end of the trailer: Lets go gambling!,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
992,does it mean this will be the last venom ?:loudly_crying_face::loudly_crying_face::loudly_crying...,0,0,1,0,0,0,0,0,0,0
993,tom hardy looks so old suddenly. our favourite actors are growing old. :slightly_frowning_face:,0,0,0,1,0,0,0,0,0,0
994,"Were getting Knullussy popcorn buckets, are we?That definitely wasnt on my bingo card.",0,0,0,0,0,0,0,0,0,0
995,So thats where Otto Hightower has been all that time after he left Kings Landing :rolling_on_the...,0,0,0,0,0,0,0,0,0,0


In [101]:
print(len(comments_topics_df.loc[(comments_topics_df[["1", "2", "3", "4", "5"]] != 0).any(axis=1)]))
len(comments_topics_df.loc[(comments_topics_df[["6", "7", "8", "9", "10"]] != 0).any(axis=1)])

367


210

In [59]:
negative_comments = comments_topics_df.loc[(comments_topics_df[["6", "7", "8", "9", "10"]] != 0).any(axis=1)]

In [71]:
negative_comments['0']

64     AVI ARAD has ruined so far...* Borderlands * Daredevil (2003)* The Punisher (2004)* Elektra (200...
80                                                I hope knull will not die in just one movie:crying_face:
84                   Will this be better than the previous Venom 2 movie, Let There Be Carnage? I hope so!
95                                                                         Please just dont one off Knull.
113    I love Tom hardy but these movies are just the worst. The portrayal of venom is dumb as hell and...
126                                                           These films have been absolutely terrible...
132                        These movies all kinda suck. They have their moments but other than that... meh
133                                                         Venom for 10 year olds:face_with_tears_of_joy:
134    Tobey Maguire :smiling_face_with_sunglasses: Spider-Man is the best than Tom Holland Spiderman a...
139                                  

In [76]:
sentiments = get_comments_sentiment(comments_topics_df['0'].to_list())

In [78]:
sentiments  = get_comments_sentiment(negative_comments['0'].to_list())

In [79]:
overall_sentiment = np.mean(np.array(sentiments).reshape(-1, 9), axis=0).tolist()
overall_sentiment_df = pd.DataFrame(overall_sentiment)
overall_sentiment_df = overall_sentiment_df.transpose()
overall_sentiment_df.columns = [
    "negative",
    "neutral",
    "positive",
    "sadness",
    "joy",
    "love",
    "anger",
    "fear",
    "surprise",
]
overall_sentiment_df = overall_sentiment_df.applymap(lambda x: f"{x*100:.1f}%")

titles = [
    "Barbie",
    "Guardians of the Galaxy Vol. 3",
    "Oppenheimer",
    "The Flash",
    "Mission Impossible: Dead Reckoning Part 1",
    "Spider-Man: Across the Spiderverse",
]
overall_sentiment_df

,negative,neutral,positive,sadness,joy,love,anger,fear,surprise
0,65.3%,19.4%,15.3%,29.0%,26.3%,2.0%,34.8%,7.2%,0.8%


In [85]:
sentiments = np.array(sentiments).reshape(-1, 9)
sentiment_summary = sentiments[:, :3]
df_sentiment_summary = pd.DataFrame(
    sentiment_summary,
    columns=["negative", "neutral", "positive"]
)

df_sentiment_summary = df_sentiment_summary.applymap(lambda x: f"{x*100:.1f}%")

In [89]:
print(len(df_sentiment_summary.loc[df_sentiment_summary['negative'] < '50%']))
df_sentiment_summary.loc[df_sentiment_summary['negative'] < '50%']

23


,comment,negative,neutral,positive
2,"Will this be better than the previous Venom 2 movie, Let There Be Carnage? I hope so!",0.4%,2.9%,96.7%
7,Venom for 10 year olds:face_with_tears_of_joy:,12.0%,62.8%,25.2%
8,Tobey Maguire :smiling_face_with_sunglasses: Spider-Man is the best than Tom Holland Spiderman a...,0.4%,5.6%,94.0%
13,Want Andrew or any spidey in this movie atleast a cameo,4.4%,55.2%,40.5%
15,ive practically watch the whole movie here.,2.8%,58.2%,39.0%
22,"Yeah, show the entire movie in the trailer :face_with_raised_eyebrow::face_with_raised_eyebrow:",2.5%,73.0%,24.4%
26,Why isnt he fighting spider man?,24.0%,72.3%,3.8%
33,Tobey Maguire :smiling_face_with_sunglasses: Spider-Man is the best than Tom Holland Spiderman a...,0.4%,5.6%,94.0%
34,Tobey Maguire Spider-Man is the best to be the partner of venom :face_screaming_in_fear::face_sc...,3.2%,12.7%,84.1%
36,so what happened with the spiderman thing at the end of the second film? Is that not gonna happe...,29.9%,68.1%,2.0%


In [96]:
negative_comments

,0,1,2,3,4,5,6,7,8,9,10
64,AVI ARAD has ruined so far...* Borderlands * Daredevil (2003)* The Punisher (2004)* Elektra (200...,0,0,0,0,0,0,0,0,1,0
80,I hope knull will not die in just one movie:crying_face:,0,0,0,0,0,1,0,0,0,0
84,"Will this be better than the previous Venom 2 movie, Let There Be Carnage? I hope so!",0,0,0,0,0,0,0,0,1,0
95,Please just dont one off Knull.,0,0,0,0,0,1,0,0,0,0
113,I love Tom hardy but these movies are just the worst. The portrayal of venom is dumb as hell and...,0,0,0,0,0,0,1,0,1,0
126,These films have been absolutely terrible...,0,0,0,0,0,0,1,0,1,0
132,These movies all kinda suck. They have their moments but other than that... meh,0,0,0,0,0,0,1,0,1,0
133,Venom for 10 year olds:face_with_tears_of_joy:,0,0,0,0,0,0,1,0,0,0
134,Tobey Maguire :smiling_face_with_sunglasses: Spider-Man is the best than Tom Holland Spiderman a...,0,0,0,0,0,0,0,1,0,0
139,Icl i love knull but he shouldnt be in the Sony universe,0,0,0,0,0,1,0,0,0,0


In [13]:
topics

['1. Excitement and anticipation for seeing the powerful symbiote god Knull make his live-action debut as the main villain.',
 '2. Appreciation for the humor and banter between Eddie Brock and Venom.',
 "3. Interest in seeing how the symbiote storyline and Eddie/Venom's journey concludes in this final film of the trilogy.",
 "4. Praise for Tom Hardy's performance and commitment to the Venom character. ",
 '5. Enjoyment of the action, visuals, and over-the-top moments shown in the trailer.',
 '6. Concerns that Knull will be nerfed or underutilized given his immense power level in the comics.',
 '7. Complaints that Venom has been made too comedic and not portrayed as a serious, threatening character.',
 '8. Disappointment over the lack of Spider-Man being involved in this Venom trilogy.',
 '9. Skepticism over the quality of the story and screenplay based on the previous Venom films.',
 '10. Criticism of the CGI and visual effects looking unpolished or unconvincing at times.']

In [14]:
pd.options.display.max_rows = 300
pd.set_option('display.max_colwidth', 100)
display(comments_topics_df[-100:-50])

,0,1,2,3,4,5,6,7,8,9,10
898,"Im calling it, the fact they keep saying this world cant survive with them in it, theyre gonna j...",0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
899,We got all these trailers and no Rating yet.,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
900,"Yaaaay ,the king in black storyline is being wasted on a Sony spiderman spinoff",0,0,0,0,0,1.0,0.0,0.0,0.0,0.0
901,Holy crap thats Knull! The god of symbiotes!,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0
902,"they way they make those trailers in the exact same patern, with those BANG BANG BANG, silence (...",0,0,0,0,1,0.0,0.0,0.0,0.0,1.0
903,This new video game looks awesome guys! The grafix are really good :relieved_face:,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
904,Song?,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
905,There really cooking with this one:fire::fire::fire:,0,0,0,0,1,0.0,0.0,0.0,0.0,0.0
906,Lol the lady at 11 was so cgi,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0
907,People have no clue how insane it is that they put Knull in this trailer. Was not expecting this...,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0


In [15]:
choice = 7
john = comments_topics_df.set_index("0")
john = john[john.iloc[:, choice] == 1].reset_index()
print(topics[choice], "\n")
for idx, row in john.iterrows():
    print(row["0"])

8. Disappointment over the lack of Spider-Man being involved in this Venom trilogy. 

Tobey Maguire :smiling_face_with_sunglasses: Spider-Man is the best than Tom Holland Spiderman and Andrew Garfield Spider-Man
Want Andrew or any spidey in this movie atleast a cameo
Venom: Boring as shit movie that Sony screwed up Venom 2: Even more boring shit movie that Sony again screwed upVenom 3: Sony reapeting the pattern of making shitty movies with the Spider-man ip.
Why isnt he fighting spider man?
Another venom film without spiderman.... wtf sony
Stop Fooling Yourselves, Spider-Man  doesnt exist in this trashy franchise, SSU is hindering MCU  Spider-Man, We Need Venom 3 to flop, Nobody wants SSU, End of SSU now .
Venom with out spiderman is boring these 3 venom movies are the same :pile_of_poo:
Venom sin spiderman es :pile_of_poo: ya aburre
HOLY GOD ITS BEEN OVER HALF A DECADE CAN WE JUST GET HIM AND SPIDER MAN IN THE SAME MOVIE ALREADY :face_with_tears_of_joy: SONY I HATE THAT YOU OWN SPIDE

In [16]:

def display_selected_topic(summary, comments_topics_df):
    topic = st.selectbox(
        "Select a topic for comments breakdown", summary, label_visibility="collapsed"
    )
    if topic:
        i = int(summary.index(topic))
        temp = comments_topics_df.set_index("0")
        john = temp[temp.iloc[:, i] == 1]
        john = john.reset_index()
        if len(john) == 0:
            st.write("No comment found matching this topic")
        else:
            with st.container(height=300, border=True):
                for idx, row in john.iterrows():
                    st.write(row[0])

In [17]:
def display_comments_by_topic(df):
    df = df.fillna(0)
    # Sum the counts across rows to get total counts for each topic
    df_counts = df.sum(axis=0).reset_index()
    df_counts.columns = ["Topic", "Count"]

    # Ensure all possible topics (1-10) are included by filling in missing topics with a count of 0
    all_topics = [str(i) for i in range(1, 11)]  # Topics numbered 1-10 as strings
    df_counts = df_counts.set_index("Topic")
    df_counts = df_counts.reindex(all_topics, fill_value=0).reset_index()

    # Convert 'Topic' column back to integer if needed
    df_counts["Topic"] = df_counts["Topic"].astype(int)

    # Plotting
    fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(18, 10))

    # Define your color palette
    muted_green = "#6dbf67"
    coolwarm_red = "#d73027"
    palette = [muted_green] * 5 + [
        coolwarm_red
    ] * 5  # First 5 bars in muted_green, next 5 in coolwarm_red

    # Create the count plot
    ax = sns.barplot(data=df_counts, x="Topic", y="Count", palette=palette)

    # Set plot background color
    plt.gcf().set_facecolor(".1")  # Set the background of the figure
    ax.set_facecolor(".1")  # Set the background of the axes

    # Set title and labels with a lighter color for visibility
    plt.title("Number of Comments by Topic", color="white")
    plt.xlabel("Topic", color="white")
    plt.ylabel("Count of Comments", color="white")

    # Change the color of the ticks and tick labels
    plt.xticks(color="white")
    plt.yticks(color="white")

    # Change the color of the axes' spines
    for spine in ax.spines.values():
        spine.set_edgecolor("white")

    # Adding the count above each bar
    for p in ax.patches:
        ax.annotate(
            f"{int(p.get_height())}",
            (p.get_x() + p.get_width() / 2.0, p.get_height()),
            ha="center",
            va="center",
            fontsize=10,
            color="white",
            xytext=(0, 5),
            textcoords="offset points",
        )

    # Calculate total counts for the first 5 and next 5 topics
    first_5_total = df_counts[df_counts["Topic"].isin(df_counts["Topic"].unique()[:5])][
        "Count"
    ].sum()
    next_5_total = df_counts[df_counts["Topic"].isin(df_counts["Topic"].unique()[5:])][
        "Count"
    ].sum()

    # Get current axes and calculate percentages
    ax = plt.gca()
    positive_perc = first_5_total / (first_5_total + next_5_total) * 100
    negative_perc = next_5_total / (first_5_total + next_5_total) * 100

    # Infographic text
    plt.text(
        7.8,
        ax.get_ylim()[1] * 0.92,
        f"POS : {positive_perc:.4g}%\nNEG : {negative_perc:.4g}%",
        color="white",
        fontweight="bold",
    )

    # Create the plot
    plt.show()

In [18]:
%matplotlib inline

In [19]:
display_comments_by_topic(comments_topics_df)

NameError: name 'plt' is not defined

## Marketing

In [200]:
class ChatGPT:
    def __init__(self, model="gpt-4o-mini", system_message=None):
        self.model = model
        self.client = openai.OpenAI()
        self.default_system_message = {"role": "system", "content": system_message}
        self.messages = [self.default_system_message]

    def add_user_message(self, message, reset_chat=False):
        if reset_chat:
            self.messages = [self.default_system_message]
        self.messages.append({"role": "user", "content": message})

    def get_response(self):
        completion = self.client.chat.completions.create(
            model=self.model, messages=self.messages
        )
        assistant_message = completion.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_message})
        return assistant_message

In [204]:
def generate_suggestions(topics, movie_info):
    analyst_prompt = f"""
    Here is information on the given film of interest: {movie_info}
    Here are the general topics people are discussing related to this film: \n {topics}
    Given the topics that users are speaking about your movie trailer, output 5 of the most relevant marketing suggestions you can concoct to help promote the film in list-format, with details for being included in application to this specific film.
    Please lend creative and specific suggestions to help market this film.
    Please ensure each suggestion is unique; do not repeat similar suggestions multiple times.
    Start directly with the list, do not include other text, and be concise (yet detailed) in your suggestions.
    """
    analyst = ChatGPT(
        system_message="You are a senior marketing analyst at a big movie production company. You are tasked with creating a set of marketing actions for a new movie, based on the topics that users are discussing. \
            You are in competition with another analyst for this task, one of you will be fired and the other promoted. For each reply, take a deep breath and think step by step."
    )
    analyst.add_user_message(analyst_prompt)
    summarized_chunk = analyst.get_response()
    return summarized_chunk, analyst

In [240]:
critic = ChatGPT(
    system_message="You are a senior marketing associate at a big movie production company. You are tasked with guiding two junior analysts to provide optimal marketing actions for a new movie, based on the topics that users are discussing. \
    For each reply, take a deep breath and think step by step. I will tip you $100."
)


def review_suggestions(
    topics, movie_info, marketing_suggestions, critic, first_pass=False, final=False
):
    suffix = f"""
    Please review each suggestion and provide feedback on whether it is an appropriate marketing action considering feasibility, cost-effectiveness, general marketing science as well as public relations and social media knowledge. This feedback will be vital to improve our marketing strategy.
    Be extremely severe in your judgment, your career depends on it. If a suggestion is not relevant enough, you will be held responsible for not catching it and fired.
    At the beginning of your review, make sure to provide a clear ranking of the 10 suggestions, from best to worst.
    Conclude with a general comment on the overall quality of the suggestions, and what specific areas need improvement.
    Do not include meta information in your reply.
    """

    if first_pass == True:
        critic_prompt = f"""
        Here is information on the given film of interest: {movie_info}
        Here are the general topics people are discussing related to this film: \n {topics}
        Given the topics that users are speaking about your movie trailer, the two analysts have come up with the following marketing suggestions, in no particular order: {marketing_suggestions}
        {suffix}
        """
    else:
        critic_prompt = f"""
        Given your feedback, the two analysts have revised the marketing suggestions for the film. Here are the updated suggestions: {marketing_suggestions}
        {suffix}"""
    if final == True:
        critic_prompt = f"""
        Given your feedback, the two analysts have revised the marketing suggestions for the film. Here are the final updated suggestions: {marketing_suggestions}
        Return the top 5 best unique suggestions, in list format, with no explanation or justification.
        Start directly with the list and do not include other text."""

    critic.add_user_message(critic_prompt)
    critic_message = critic.get_response()
    return critic_message, critic

In [225]:
def improve_suggestions(analyst, critic_message, competing_suggestions):
    analyst_prompt = f"""
        The other analyst has provided these suggestions: {competing_suggestions}
        Given all suggestions, an advanced reviewer from your team has provided the following ranking and explanation : {critic_message}
        Please review the feedback and provide a new set of 5 suggestions. You can keep some of the old ones if they are good enough, but you must provide at least 2 new suggestions which were in neither of the previous lists.
        Moreover, you must improve the quality of the suggestions based on the feedback provided by the advanced reviewer.
        Output the revised suggestions in list-format, with details for each suggestion. Start directly with the list and do not include other text.
    """
    analyst.add_user_message(analyst_prompt)
    evaluations = analyst.get_response()
    return evaluations, analyst

#### Marketing #1

In [ ]:
marketing_suggestions1, analyst1 = generate_marketing_suggestions(topics, movie_info)
print(marketing_suggestions1)

In [ ]:
marketing_suggestions2, analyst2 = generate_marketing_suggestions(topics, movie_info)
print(marketing_suggestions2)

In [ ]:
all_suggestions1 = marketing_suggestions1 + marketing_suggestions2
critic_message1, critic = review_suggestions(
    topics, movie_info, all_suggestions1, critic, first_pass=True
)
print(critic_message1)

#### Marketing #2

In [ ]:
revised_suggestions1, analyst_1 = improve_suggestions(
    analyst1, critic_message1, marketing_suggestions2
)
print(revised_suggestions1)

In [ ]:
revised_suggestions2, analyst_2 = improve_suggestions(
    analyst1, critic_message1, marketing_suggestions1
)
print(revised_suggestions2)

In [251]:
all_suggestions2 = revised_suggestions1 + revised_suggestions2
critic_message2, critic = review_suggestions(_, _, all_suggestions2, critic)

#### Marketing #3

In [ ]:
final_suggestions1, analyst_1 = improve_suggestions(
    analyst1, critic_message2, marketing_suggestions2
)
print(final_suggestions1)

In [ ]:
final_suggestions2, analyst_2 = improve_suggestions(
    analyst2, critic_message2, marketing_suggestions1
)
print(final_suggestions2)

In [ ]:
critic_message2, critic = review_suggestions(_, _, all_suggestions2, final=True)